In [ ]:
# Step 1: Install PySpark
# PySpark is installed using pip, Python's package manager.
pip install pyspark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Step 2: Create SparkSession
# SparkSession is the entry point to Spark functionality.
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NetflixETL") \
    .getOrCreate()

print(spark.version)

4.1.2


In [ ]:
# Step 3: Generate Users Dataset with Pandas
# Import Pandas library for creating and managing tabular data
import pandas as pd

# Import random library for generating random values
import random

# Create a list of countries that users can belong to
countries = [
    "India",
    "USA",
    "UK",
    "Canada",
    "Australia",
    "Germany",
    "France"
]

# Create a list of possible Netflix subscription plans
subscription_types = [
    "Basic",
    "Standard",
    "Premium"
]

# Create an empty list to store user records
users = []

# Generate 500 users with unique IDs
for user_id in range(1, 501):

    # Add a user record to the users list
    users.append([

        # Unique user ID
        user_id,

        # Generate user names like User_1, User_2, User_3 ...
        f"User_{user_id}",

        # Randomly assign a country from the countries list
        random.choice(countries),

        # Randomly assign a subscription plan
        random.choice(subscription_types)
    ])

# Convert the list of user records into a Pandas DataFrame
users_df = pd.DataFrame(

    # Data source
    users,

    # Define column names
    columns=[
        "user_id",             # Unique identifier for each user
        "user_name",           # User name
        "country",             # User's country
        "subscription_type"    # Netflix subscription plan
    ]
)

# Export the DataFrame to a CSV file
# index=False prevents Pandas from creating an extra index column
users_df.to_csv(
    "C:/Users/jovit/Desktop/Netflix_Project/users.csv",
    index=False
)

# Display the first 5 records for verification
print(users_df.head())

   user_id user_name    country subscription_type
0        1    User_1     Canada          Standard
1        2    User_2  Australia             Basic
2        3    User_3         UK           Premium
3        4    User_4     Canada           Premium
4        5    User_5    Germany           Premium


In [ ]:
# Step 4: Generate User Activity Dataset
# Import Pandas library for creating and manipulating DataFrames
import pandas as pd

# Import random library for generating random values
import random

# Read the Netflix dataset into a Pandas DataFrame
netflix = pd.read_csv("C:/Users/jovit/Desktop/Netflix_Project/netflix_titles_clean.csv")

# Extract all show IDs from the Netflix dataset
# Convert the show_id column into a Python list
show_ids = netflix["show_id"].tolist()

# Create an empty list to store activity records
activities = []

# Generate 5000 user activity records
for i in range(1, 5001):

    # Append a new activity record to the activities list
    activities.append([

        # Activity ID (Unique identifier for each activity)
        i,

        # Randomly assign a user ID between 1 and 500
        # Assumes we have 500 users in users.csv
        random.randint(1, 500),

        # Randomly select a show_id from the Netflix dataset
        # This creates a relationship between activity and content
        random.choice(show_ids),

        # Generate random watch time between 10 and 300 minutes
        random.randint(10, 300)
    ])

# Convert the list of activity records into a DataFrame
activity_df = pd.DataFrame(

    # Data source
    activities,

    # Define column names
    columns=[
        "activity_id",    # Unique activity identifier
        "user_id",        # User who watched the content
        "show_id",        # Content watched
        "watch_minutes"   # Minutes watched
    ]
)

# Export DataFrame to a CSV file
# index=False prevents Pandas from creating an extra index column
activity_df.to_csv(
    "C:/Users/jovit/Desktop/Netflix_Project/user_activity.csv",
    index=False
)

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NetflixETL") \
    .getOrCreate() # Create a SparkSession

print(spark.version) # Print the Spark version

4.1.2


In [ ]:
content_df = spark.read.csv(
    "C:/Users/jovit/Desktop/Netflix_Project/netflix_titles_clean.csv",
    header=True,
    inferSchema=True
) # read the netflix_titles_clean.csv file into a DataFrame

In [ ]:
content_df.printSchema() # Inspect schema

root
 |-- show_id: string (nullable = true)
 |-- show_type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)
 |-- added_year: string (nullable = true)



In [ ]:
content_df.show(5) # Inspect sample records

+-------+---------+--------------------+---------------+--------------------+-------------+----------+------------+------+---------+--------------------+--------------------+----------+
|show_id|show_type|               title|       director|           cast_name|      country|date_added|release_year|rating| duration|           listed_in|         description|added_year|
+-------+---------+--------------------+---------------+--------------------+-------------+----------+------------+------+---------+--------------------+--------------------+----------+
|     s1|    Movie|Dick Johnson Is Dead|Kirsten Johnson|       Not Available|United States|2021-09-25|        2020| PG-13|   90 min|       Documentaries|As her father nea...|    2021.0|
|     s2|  TV Show|       Blood & Water|        Unknown|Ama Qamata, Khosi...| South Africa|2021-09-24|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|    2021.0|
|     s3|  TV Show|           Ganglands|Julien Leclercq|Sami Bouajila,

In [ ]:
content_df.count() # Count the number of rows

8809

In [ ]:
len(content_df.columns) # Count the number of columns

13

In [ ]:
users_df = spark.read.csv(
    "C:/Users/jovit/Desktop/Netflix_Project/users.csv",
    header=True,
    inferSchema=True
) # read the users.csv file into a DataFrame

activity_df = spark.read.csv(
    "C:/Users/jovit/Desktop/Netflix_Project/user_activity.csv",
    header=True,
    inferSchema=True
) # read the user_activity.csv file into a DataFrame

In [ ]:
users_df.show() # Inspect sample records

+-------+---------+---------+-----------------+
|user_id|user_name|  country|subscription_type|
+-------+---------+---------+-----------------+
|      1|   User_1|   Canada|         Standard|
|      2|   User_2|Australia|            Basic|
|      3|   User_3|       UK|          Premium|
|      4|   User_4|   Canada|          Premium|
|      5|   User_5|  Germany|          Premium|
|      6|   User_6|  Germany|         Standard|
|      7|   User_7|  Germany|            Basic|
|      8|   User_8|  Germany|         Standard|
|      9|   User_9|      USA|          Premium|
|     10|  User_10|      USA|         Standard|
|     11|  User_11|      USA|         Standard|
|     12|  User_12|  Germany|          Premium|
|     13|  User_13|       UK|         Standard|
|     14|  User_14|  Germany|         Standard|
|     15|  User_15|       UK|          Premium|
|     16|  User_16|   France|         Standard|
|     17|  User_17|    India|          Premium|
|     18|  User_18|       UK|         St

In [ ]:
activity_df.show() # Inspect sample records

+-----------+-------+-------+-------------+
|activity_id|user_id|show_id|watch_minutes|
+-----------+-------+-------+-------------+
|          1|    313|  s7076|          134|
|          2|    455|  s6205|          126|
|          3|    317|  s2292|           10|
|          4|    205|  s2100|          112|
|          5|    453|  s7494|          147|
|          6|    473|   s776|          223|
|          7|    465|   s786|           70|
|          8|     63|  s4895|          233|
|          9|    160|  s6434|          161|
|         10|    447|   s690|          183|
|         11|     45|  s6340|          112|
|         12|     42|  s7678|          147|
|         13|    491|  s3915|           13|
|         14|    222|  s3548|          159|
|         15|    433|  s3899|          222|
|         16|    482|  s5272|           47|
|         17|    382|  s7699|           49|
|         18|    361|   s462|           89|
|         19|    391|  s2149|          245|
|         20|    399|   s548|   

In [ ]:
users_df.show() # Inspect sample records

+-------+---------+---------+-----------------+
|user_id|user_name|  country|subscription_type|
+-------+---------+---------+-----------------+
|      1|   User_1|   Canada|         Standard|
|      2|   User_2|Australia|            Basic|
|      3|   User_3|       UK|          Premium|
|      4|   User_4|   Canada|          Premium|
|      5|   User_5|  Germany|          Premium|
|      6|   User_6|  Germany|         Standard|
|      7|   User_7|  Germany|            Basic|
|      8|   User_8|  Germany|         Standard|
|      9|   User_9|      USA|          Premium|
|     10|  User_10|      USA|         Standard|
|     11|  User_11|      USA|         Standard|
|     12|  User_12|  Germany|          Premium|
|     13|  User_13|       UK|         Standard|
|     14|  User_14|  Germany|         Standard|
|     15|  User_15|       UK|          Premium|
|     16|  User_16|   France|         Standard|
|     17|  User_17|    India|          Premium|
|     18|  User_18|       UK|         St

In [ ]:
users_df.printSchema() # Inspect schema

root
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- subscription_type: string (nullable = true)



In [ ]:
activity_df.printSchema() # Inspect schema

root
 |-- activity_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- show_id: string (nullable = true)
 |-- watch_minutes: integer (nullable = true)

